# Image Generation Model Evaluation

This notebook evaluates different image generation models for gift card illustrations.

## Metrics Tracked
- **latency_ms**: Time per image generation
- **output_size_bytes**: Size of generated image
- **quality_score**: LLM-as-a-judge + aesthetic scoring
- **cost_per_image**: Cost per image generation
- **monthly_estimate**: Cost for 2B images

## Models Evaluated
- Recraft API
- TokenFactory / Flux API
- GPT-image-1-mini (if available)
- SDXL Lightning (self-hosted)
- Flux (self-hosted)

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.clients.image_client import ImageClient
from src.clients.llm_client import LLMClient
from src.evaluation import (
    calculate_image_monthly_cost_estimate,
    evaluate_quality_with_judge,
    log_image_evaluation,
    setup_mlflow,
)
from src.prompts import (
    generate_image_prompt,
    generate_image_quality_judge_prompt,
)
from src.test_samples import get_test_profiles, get_test_recommendations_for_image
from src.utils import load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root()

In [ ]:
setup_mlflow("image_generation_eval")

In [ ]:
test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_image()

In [ ]:
models_to_evaluate = []

if os.getenv("RECRAFT_API_KEY"):
    models_to_evaluate.append({
        "name": "recraftv3",
        "service": "recraft",
        "client": ImageClient(
            service="recraft",
            api_key=os.getenv("RECRAFT_API_KEY"),
            model="recraftv3"
        ),
        "cost_per_image": 0.01,
    })

if os.getenv("TOKEN_FACTORY_API_KEY"):
    models_to_evaluate.append({
        "name": "tokenfactory-flux",
        "service": "tokenfactory",
        "client": ImageClient(
            service="tokenfactory",
            api_key=os.getenv("TOKEN_FACTORY_API_KEY"),
            model="flux"
        ),
        "cost_per_image": 0.005,
    })

sdxl_url = os.getenv("SDXL_LIGHTNING_URL", "http://localhost:7860")
if sdxl_url:
    models_to_evaluate.append({
        "name": "sdxl-lightning",
        "service": "sdxl",
        "client": ImageClient(
            service="sdxl",
            base_url=sdxl_url,
            model="sdxl-lightning"
        ),
        "cost_per_image": 0.0,
    })

flux_url = os.getenv("FLUX_SELF_HOSTED_URL", "http://localhost:7861")
if flux_url:
    models_to_evaluate.append({
        "name": "flux-dev",
        "service": "flux",
        "client": ImageClient(
            service="flux",
            base_url=flux_url,
            model="flux-dev"
        ),
        "cost_per_image": 0.0,
    })

In [ ]:
judge_client = None
if os.getenv("OPENAI_API_KEY"):
    judge_client = LLMClient(
        base_url="https://api.openai.com/v1",
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-3.5-turbo"
    )
elif os.getenv("TOKEN_FACTORY_API_KEY") and os.getenv("TOKEN_FACTORY_BASE_URL"):
    judge_client = LLMClient(
        base_url=os.getenv("TOKEN_FACTORY_BASE_URL"),
        api_key=os.getenv("TOKEN_FACTORY_API_KEY"),
        model="meta-llama/Meta-Llama-3-8B-Instruct"
    )

In [ ]:
results = []

for model_config in models_to_evaluate:
    model_name = model_config["name"]
    service = model_config["service"]
    client = model_config["client"]

    for kid_profile, gift_recommendation in zip(test_profiles, test_recommendations):
        prompt = generate_image_prompt(kid_profile, gift_recommendation)

        try:
            if service == "recraft":
                image_url, metrics = client.generate(
                    prompt=prompt,
                    style="digital_illustration",
                    size="1024x1434",
                    extra_body={"substyle": "hand_drawn"}
                )
            else:
                image_url, metrics = client.generate(
                    prompt=prompt,
                    size="1024x1024"
                )

            quality_score = None
            if judge_client:
                judge_prompt = generate_image_quality_judge_prompt(prompt, image_url, kid_profile)
                quality_score = evaluate_quality_with_judge(judge_client, judge_prompt)

            run_id = log_image_evaluation(
                model_name=model_name,
                service=service,
                prompt=prompt,
                image_url=image_url,
                metrics=metrics,
                quality_score=quality_score,
                tags={
                    "kid_name": kid_profile.name,
                    "kid_age": str(kid_profile.age),
                    "task": "image_generation",
                }
            )

            monthly_cost = calculate_image_monthly_cost_estimate(
                cost_per_image=model_config["cost_per_image"],
            )

            results.append({
                "model": model_name,
                "service": service,
                "kid_name": kid_profile.name,
                "kid_age": kid_profile.age,
                "latency_ms": metrics.get("latency_ms", 0),
                "output_size_bytes": metrics.get("output_size_bytes"),
                "quality_score": quality_score,
                "cost_per_image": model_config["cost_per_image"],
                "monthly_cost_usd": monthly_cost,
                "run_id": run_id,
            })

        except Exception as e:
            results.append({
                "model": model_name,
                "service": service,
                "kid_name": kid_profile.name,
                "error": str(e),
            })

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)
print("Evaluation Results Summary:")
print(df_results.groupby("model").agg({
    "latency_ms": "mean",
    "quality_score": "mean",
    "cost_per_image": "mean",
    "monthly_cost_usd": "mean",
}).round(2))